# 01 - Data EDA and Assumptions

This notebook explores the raw dataset before solving the exercises.

Purpose:

1. inspect raw rows and date parsing
2. measure timestamp quality and time coverage
3. measure missing IDs and names
4. test whether missing IDs can be safely inferred from exact string matches
5. document the assumptions carried into the solution notebooks

Recommended next notebook: `02_exercise_2_walkthrough.ipynb`.


## Functions used in this notebook

### `create_spark_session(app_name, master='local[*]', shuffle_partitions=64)`
- **Description:** Starts the Spark environment used for both raw-file inspection and normalized-data analysis.
- **Input:** Application name and optional Spark settings.
- **Output:** A configured `SparkSession`.
- **Why this operation is selected for efficiency:** Reusing the same Spark setup avoids environment drift between EDA and the final solution, so the reviewer is inspecting the same engine used in production notebooks.

### `resolve_lastfm_input_path(project_root)`
- **Description:** Finds the downloaded TSV under `data/raw/`.
- **Input:** Repository root path.
- **Output:** `Path` to the raw TSV file.
- **Why this operation is selected for efficiency:** It keeps the EDA portable and avoids manual path edits.

### `LASTFM_SCHEMA`
- **Description:** Explicit Spark schema for the raw TSV columns.
- **Input:** Used as the schema argument of `spark.read.csv(...)`.
- **Output:** A typed raw DataFrame with stable column names.
- **Why this operation is selected for efficiency:** Providing the schema avoids Spark schema inference on a very large TSV, which is slower and can introduce type inconsistencies.

### `load_lastfm_events(...)`
- **Description:** Loads the normalized event table for EDA on the cleaned analytical dataset.
- **Input:** Spark session and project root.
- **Output:** Normalized Spark DataFrame with parsed timestamps.
- **Why this operation is selected for efficiency:** It lets the notebook compare raw-file quality questions with the normalized dataset without duplicating ingestion logic.

### Why both raw TSV and normalized Parquet are inspected here
- **Description:** The raw TSV is used to inspect original null patterns and timestamp parsing risk; the normalized dataset is used to inspect the exact data the downstream solution consumes.
- **Input:** Raw TSV and staged Parquet.
- **Output:** Evidence-based assumptions for IDs, names, and dates.
- **Why this operation is selected for efficiency:** The EDA separates data-quality questions from production computation, which keeps the final solution lean while still documenting the reasoning clearly.


In [1]:
from pathlib import Path
import logging
import sys

logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(name)s:%(message)s')
logger = logging.getLogger('lastfm.notebook')

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


WindowsPath('C:/Users/GonzaloFigueroa/Documents/Private/BME/lastfm-analysis')

In [2]:
import pandas as pd
from pyspark.sql import functions as F

from src.spark_utils import create_spark_session
from src.sessionization import (
    ensure_lastfm_dataset_available,
    LASTFM_SCHEMA,
    load_lastfm_events,
    resolve_lastfm_input_path,
    resolve_lastfm_parquet_path,
    stage_lastfm_events_to_parquet,
)


In [3]:
spark = create_spark_session(app_name='01-data-eda-and-assumptions')
spark


In [4]:
raw_input_path = ensure_lastfm_dataset_available(PROJECT_ROOT)
parquet_path = resolve_lastfm_parquet_path(PROJECT_ROOT)

logger.info(f'Raw TSV path: {raw_input_path}')
logger.info(f'Prepared Parquet path: {parquet_path}')


INFO:lastfm.notebook:Raw TSV path: C:\Users\GonzaloFigueroa\Documents\Private\BME\lastfm-analysis\data\raw\lastfm-dataset-1K\userid-timestamp-artid-artname-traid-traname.tsv


INFO:lastfm.notebook:Prepared Parquet path: C:\Users\GonzaloFigueroa\Documents\Private\BME\lastfm-analysis\data\processed\lastfm_events_parquet


In [5]:
raw_lastfm_df = (
    spark.read
    .option('sep', '	')
    .option('header', False)
    .schema(LASTFM_SCHEMA)
    .csv(str(raw_input_path))
)

raw_lastfm_df.show(10, truncate=False)


+-----------+--------------------+------------------------------------+-----------+--------+------------------------------------------+
|user_id    |started_at_raw      |artist_id                           |artist_name|track_id|track_name                                |
+-----------+--------------------+------------------------------------+-----------+--------+------------------------------------------+
|user_000001|2009-05-04T23:08:57Z|f1b1cf71-bd35-4e99-8624-24a6e15f133a|Deep Dish  |NULL    |Fuck Me Im Famous (Pacha Ibiza)-09-28-2007|
|user_000001|2009-05-04T13:54:10Z|a7f7df4a-77d8-4f12-8acd-5c60c93f4de8|坂本龍一   |NULL    |Composition 0919 (Live_2009_4_15)         |
|user_000001|2009-05-04T13:52:04Z|a7f7df4a-77d8-4f12-8acd-5c60c93f4de8|坂本龍一   |NULL    |Mc2 (Live_2009_4_15)                      |
|user_000001|2009-05-04T13:42:52Z|a7f7df4a-77d8-4f12-8acd-5c60c93f4de8|坂本龍一   |NULL    |Hibari (Live_2009_4_15)                   |
|user_000001|2009-05-04T13:42:11Z|a7f7df4a-77d8-4f12-8acd-5c

In [6]:
raw_lastfm_df.printSchema()


root
 |-- user_id: string (nullable = true)
 |-- started_at_raw: string (nullable = true)
 |-- artist_id: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)



## Date quality and coverage

The next cells focus specifically on the date field:

- how many raw rows can be parsed into timestamps
- the minimum and maximum dates in the usable data
- how events are distributed across years and months


In [7]:
parsed_date_df = raw_lastfm_df.withColumn(
    'started_at',
    F.to_timestamp('started_at_raw', "yyyy-MM-dd'T'HH:mm:ss'Z'")
)

parsed_date_df.select('started_at_raw', 'started_at').show(10, truncate=False)


+--------------------+-------------------+
|started_at_raw      |started_at         |
+--------------------+-------------------+
|2009-05-04T23:08:57Z|2009-05-04 23:08:57|
|2009-05-04T13:54:10Z|2009-05-04 13:54:10|
|2009-05-04T13:52:04Z|2009-05-04 13:52:04|
|2009-05-04T13:42:52Z|2009-05-04 13:42:52|
|2009-05-04T13:42:11Z|2009-05-04 13:42:11|
|2009-05-04T13:38:31Z|2009-05-04 13:38:31|
|2009-05-04T13:33:28Z|2009-05-04 13:33:28|
|2009-05-04T13:23:45Z|2009-05-04 13:23:45|
|2009-05-04T13:19:22Z|2009-05-04 13:19:22|
|2009-05-04T13:13:38Z|2009-05-04 13:13:38|
+--------------------+-------------------+
only showing top 10 rows


In [8]:
raw_quality_summary = parsed_date_df.agg(
    F.count('*').alias('raw_rows'),
    F.sum(F.when(F.col('started_at').isNull(), 1).otherwise(0)).alias('invalid_timestamp_rows'),
    F.sum(F.when(F.col('track_name').isNull(), 1).otherwise(0)).alias('null_track_name_rows_raw'),
    F.sum(F.when(F.col('artist_name').isNull(), 1).otherwise(0)).alias('null_artist_name_rows_raw'),
).first().asDict()

pd.DataFrame([raw_quality_summary])


,raw_rows,invalid_timestamp_rows,null_track_name_rows_raw,null_artist_name_rows_raw
0,19150868,0,1,0


In [9]:
parquet_path = stage_lastfm_events_to_parquet(spark, PROJECT_ROOT)
events_df = load_lastfm_events(spark, PROJECT_ROOT)

logger.info(f'Parquet dataset ready at: {parquet_path}')


INFO:lastfm.notebook:Parquet dataset ready at: C:\Users\GonzaloFigueroa\Documents\Private\BME\lastfm-analysis\data\processed\lastfm_events_parquet


In [10]:
date_summary = events_df.agg(
    F.count('*').alias('normalized_rows'),
    F.min('started_at').alias('min_started_at'),
    F.max('started_at').alias('max_started_at'),
    F.countDistinct(F.year('started_at')).alias('distinct_years'),
    F.countDistinct(F.date_trunc('month', 'started_at')).alias('distinct_months'),
).first().asDict()

pd.DataFrame([date_summary])


,normalized_rows,min_started_at,max_started_at,distinct_years,distinct_months
0,19150867,2005-02-14 01:00:07,2013-09-29 20:32:04,7,55


In [11]:
yearly_events_df = (
    events_df.groupBy(F.year('started_at').alias('year'))
    .agg(F.count('*').alias('play_count'))
    .orderBy('year')
)

yearly_events_df.show(20, truncate=False)


+----+----------+
|year|play_count|
+----+----------+
|2005|1070656   |
|2006|4255308   |
|2007|5358216   |
|2008|5929147   |
|2009|2537538   |
|2010|1         |
|2013|1         |
+----+----------+



In [12]:
monthly_events_df = (
    events_df.groupBy(F.date_trunc('month', 'started_at').alias('month_start'))
    .agg(F.count('*').alias('play_count'))
    .orderBy('month_start')
)

monthly_events_df.show(24, truncate=False)


+-------------------+----------+
|month_start        |play_count|
+-------------------+----------+
|2005-02-01 00:00:00|24690     |
|2005-03-01 00:00:00|49273     |
|2005-04-01 00:00:00|74085     |
|2005-05-01 00:00:00|69549     |
|2005-06-01 00:00:00|81691     |
|2005-07-01 00:00:00|87337     |
|2005-08-01 00:00:00|100275    |
|2005-09-01 00:00:00|109860    |
|2005-10-01 00:00:00|135415    |
|2005-11-01 00:00:00|141494    |
|2005-12-01 00:00:00|196987    |
|2006-01-01 00:00:00|239392    |
|2006-02-01 00:00:00|245544    |
|2006-03-01 00:00:00|309748    |
|2006-04-01 00:00:00|321632    |
|2006-05-01 00:00:00|360926    |
|2006-06-01 00:00:00|351990    |
|2006-07-01 00:00:00|374439    |
|2006-08-01 00:00:00|392334    |
|2006-09-01 00:00:00|380928    |
|2006-10-01 00:00:00|405162    |
|2006-11-01 00:00:00|421984    |
|2006-12-01 00:00:00|451229    |
|2007-01-01 00:00:00|472480    |
+-------------------+----------+
only showing top 24 rows


## ID and name quality

The next cells measure missing values and ambiguity in the identifier columns.


In [13]:
missing_summary = events_df.agg(
    F.count('*').alias('total_rows'),
    F.sum(F.when(F.col('artist_id').isNull(), 1).otherwise(0)).alias('null_artist_id_rows'),
    F.sum(F.when(F.col('track_id').isNull(), 1).otherwise(0)).alias('null_track_id_rows'),
    F.sum(F.when(F.col('artist_name').isNull(), 1).otherwise(0)).alias('null_artist_name_rows'),
    F.sum(F.when(F.col('track_name').isNull(), 1).otherwise(0)).alias('null_track_name_rows'),
    F.countDistinct('artist_id').alias('distinct_artist_ids'),
    F.countDistinct('artist_name').alias('distinct_artist_names'),
    F.countDistinct('track_id').alias('distinct_track_ids'),
    F.countDistinct('track_name').alias('distinct_track_names'),
).first().asDict()

pd.DataFrame([missing_summary])


,total_rows,null_artist_id_rows,null_track_id_rows,null_artist_name_rows,null_track_name_rows,distinct_artist_ids,distinct_artist_names,distinct_track_ids,distinct_track_names
0,19150867,602165,2168587,0,0,107397,174089,961416,1084749


In [14]:
events_df.filter(F.col('track_id').isNull()).select(
    'user_id', 'started_at', 'artist_id', 'artist_name', 'track_id', 'track_name'
).show(20, truncate=False)


+-----------+-------------------+------------------------------------+-----------------------------+--------+------------------------------------------------------------------+
|user_id    |started_at         |artist_id                           |artist_name                  |track_id|track_name                                                        |
+-----------+-------------------+------------------------------------+-----------------------------+--------+------------------------------------------------------------------+
|user_000013|2006-12-22 18:12:00|NULL                                |Subwoofer                    |NULL    |Vaporize                                                          |
|user_000013|2006-12-22 18:56:46|4aae17a7-9f0c-487b-b60e-f8eafb410b1d|Nick Cave                    |NULL    |Far From Me (Live Mtv 1997)                                       |
|user_000013|2006-12-22 19:00:54|4aae17a7-9f0c-487b-b60e-f8eafb410b1d|Nick Cave                    |NULL    |Up Jum

In [15]:
events_df.filter(F.col('artist_id').isNull()).select(
    'user_id', 'started_at', 'artist_id', 'artist_name', 'track_id', 'track_name'
).show(20, truncate=False)


+-----------+-------------------+---------+---------------------------------+--------+--------------------+
|user_id    |started_at         |artist_id|artist_name                      |track_id|track_name          |
+-----------+-------------------+---------+---------------------------------+--------+--------------------+
|user_000013|2006-12-22 18:12:00|NULL     |Subwoofer                        |NULL    |Vaporize            |
|user_000013|2006-12-22 19:50:47|NULL     |Cuppycake                        |NULL    |You'Re My Honeybunch|
|user_000013|2006-12-23 23:59:26|NULL     |Dj Boros                         |NULL    |Track  9            |
|user_000013|2006-12-28 13:23:56|NULL     |:Wumpscut: (By :Wumpscut:)       |NULL    |Don'T Go            |
|user_000013|2006-12-30 19:06:44|NULL     |Maynard, Mike Patton, Rage       |NULL    |They Call Me Dr Love|
|user_000013|2006-12-30 19:13:16|NULL     |Maynard, Mike Patton, Rage       |NULL    |They Call Me Dr Love|
|user_000013|2006-12-30 19:2

In [16]:
track_map_df = (
    events_df.filter(F.col('track_id').isNotNull())
    .groupBy('artist_name', 'track_name')
    .agg(
        F.countDistinct('track_id').alias('distinct_track_ids_for_name_pair'),
        F.first('track_id').alias('example_track_id'),
    )
)

track_map_stats = track_map_df.agg(
    F.sum(F.when(F.col('distinct_track_ids_for_name_pair') == 1, 1).otherwise(0)).alias('unique_name_pairs'),
    F.sum(F.when(F.col('distinct_track_ids_for_name_pair') > 1, 1).otherwise(0)).alias('ambiguous_name_pairs'),
).first().asDict()

pd.DataFrame([track_map_stats])


,unique_name_pairs,ambiguous_name_pairs
0,952350,4533


In [17]:
track_map_df.filter(F.col('distinct_track_ids_for_name_pair') > 1).orderBy(
    F.desc('distinct_track_ids_for_name_pair'), F.asc('artist_name'), F.asc('track_name')
).show(20, truncate=False)


+-----------------------+--------------------------------------------------------------------------------------+--------------------------------+------------------------------------+
|artist_name            |track_name                                                                            |distinct_track_ids_for_name_pair|example_track_id                    |
+-----------------------+--------------------------------------------------------------------------------------+--------------------------------+------------------------------------+
|U2                     |I Still Haven'T Found What I'M Looking For                                            |6                               |94793531-2679-4ea7-a273-945fdf64c06f|
|Buzzcocks              |Ever Fallen In Love (With Someone You Shouldn'T'Ve)                                   |5                               |5b2e6512-b941-40e8-970d-861a8560e537|
|Pink Floyd             |Another Brick In The Wall, Part 2                           

In [18]:
backfillable_track_rows = (
    events_df.filter(F.col('track_id').isNull()).alias('n')
    .join(
        track_map_df.filter(F.col('distinct_track_ids_for_name_pair') == 1)
        .select('artist_name', 'track_name', 'example_track_id')
        .alias('m'),
        on=['artist_name', 'track_name'],
        how='inner',
    )
    .count()
)

pd.DataFrame([{
    'null_track_id_rows': int(missing_summary['null_track_id_rows']),
    'exact_backfillable_null_track_rows': int(backfillable_track_rows),
    'exact_backfillable_rate_pct_of_null_track_rows': round(100.0 * backfillable_track_rows / missing_summary['null_track_id_rows'], 4),
}])


,null_track_id_rows,exact_backfillable_null_track_rows,exact_backfillable_rate_pct_of_null_track_rows
0,2168587,421,0.0194


In [19]:
artist_map_df = (
    events_df.filter(F.col('artist_id').isNotNull())
    .groupBy('artist_name')
    .agg(
        F.countDistinct('artist_id').alias('distinct_artist_ids_for_name'),
        F.first('artist_id').alias('example_artist_id'),
    )
)

artist_map_stats = artist_map_df.agg(
    F.sum(F.when(F.col('distinct_artist_ids_for_name') == 1, 1).otherwise(0)).alias('unique_artist_names'),
    F.sum(F.when(F.col('distinct_artist_ids_for_name') > 1, 1).otherwise(0)).alias('ambiguous_artist_names'),
).first().asDict()

pd.DataFrame([artist_map_stats])


,unique_artist_names,ambiguous_artist_names
0,102308,2295


In [20]:
artist_map_df.filter(F.col('distinct_artist_ids_for_name') > 1).orderBy(
    F.desc('distinct_artist_ids_for_name'), F.asc('artist_name')
).show(20, truncate=False)


+----------------+----------------------------+------------------------------------+
|artist_name     |distinct_artist_ids_for_name|example_artist_id                   |
+----------------+----------------------------+------------------------------------+
|Aurora          |7                           |19e659b1-2a5a-4f7e-918d-0df968ff6d44|
|H2O             |7                           |6b91614f-7e0c-4d03-8d44-af111fe6fd6a|
|Velvet          |7                           |738d04ca-da4d-4d56-b7a7-f80bc9e3dd63|
|Angel           |6                           |b6fe8845-2613-413f-9767-e6083b108059|
|Bliss           |6                           |58f10cb9-7d01-4dbd-bd38-b5b943bba7b6|
|Brainstorm      |6                           |ba089de9-7821-44f8-8c26-ccc81f325062|
|Naomi           |6                           |6764b623-b1d6-4c65-960a-917d79bda62b|
|Neon            |6                           |f841834f-31bb-47c4-bdf9-31b630c2a28e|
|Odyssey         |6                           |38746824-7adc-4dac

In [21]:
backfillable_artist_rows = (
    events_df.filter(F.col('artist_id').isNull()).alias('n')
    .join(
        artist_map_df.filter(F.col('distinct_artist_ids_for_name') == 1)
        .select('artist_name', 'example_artist_id')
        .alias('m'),
        on=['artist_name'],
        how='inner',
    )
    .count()
)

pd.DataFrame([{
    'null_artist_id_rows': int(missing_summary['null_artist_id_rows']),
    'exact_backfillable_null_artist_rows': int(backfillable_artist_rows),
    'exact_backfillable_rate_pct_of_null_artist_rows': round(100.0 * backfillable_artist_rows / missing_summary['null_artist_id_rows'], 4),
}])


,null_artist_id_rows,exact_backfillable_null_artist_rows,exact_backfillable_rate_pct_of_null_artist_rows
0,602165,1248,0.2073


## Assumptions carried into the solution notebooks

Based on the EDA above, the solution uses these assumptions:

1. A session starts when the gap between consecutive track start times for the same user is strictly greater than 20 minutes.
2. The pipeline keeps the original IDs exactly as provided by the dataset.
3. The solution does **not** use fuzzy matching to infer missing IDs from similar strings.
4. Exact deterministic ID backfilling was tested and found to recover only a tiny share of missing IDs, so it was not adopted as a core preprocessing step.
5. For Exercise 2, the final song grouping uses exact `artist_name + track_name` so rows with missing `track_id` are still represented.
6. The production notebooks use Parquet plus Spark `session_window` so the heavy computations scale better than repeatedly reading the TSV or materializing a full row-level session table.

Recommended next notebook: `02_exercise_2_walkthrough.ipynb`.


In [22]:
spark.stop()
